In [1]:
!pip install --upgrade pyarrow


   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.6 MB ? eta -:--:--
    --------------------------------------- 0.5/28.6 MB 1.4 MB/s eta 0:00:21
   - -------------------------------------- 0.8/28.6 MB 1.5 MB/s eta 0:00:18
   - -------------------------------------- 1.3/28.6 MB 1.8 MB/s eta 0:00:16
   -- ------------------------------------- 1.6/28.6 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 2.1/28.6 MB 1.8 MB/s eta 0:00:15
   --- ------------------------------------ 2.6/28.6 MB 2.1 MB/s eta 0:00:13
   ---- ----------------------------------- 3.1/28.6 MB 2.0 MB/s eta 0:00:13
   ----- ---------------------------------- 3.7/28.6 MB 2.1 MB/s eta 0:00:12
   ------ --------------------------------- 4.5/28.6 MB 2.3 MB/s eta 0:00:11
   ------ --------------------------------- 5.0/28.6 MB 2.3 MB/s eta 0:00:11
   -------- ------------------------------- 5.8/28.6 MB 2.4 MB/s eta 0:00:10
   -------- -

In [2]:
import os
import re
import numpy as np
import pandas as pd
import Levenshtein
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Create required output directory
os.makedirs("output", exist_ok=True)

# Load Training Data
train_s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
train_gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")

# Load Test Data
test_s1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_s2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_s3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")

In [4]:
def clean_text_pure_python(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\\b(corporation|corp)\\b', 'corp', text)
    text = re.sub(r'\\b(private|pvt)\\b', 'pvt', text)
    text = re.sub(r'\\b(limited|ltd)\\b', 'ltd', text)
    text = re.sub(r'\\s+and\\s+', ' & ', text)
    text = re.sub(r'[^a-z0-9\\s&]', '', text)
    return re.sub(r'\\s+', ' ', text).strip()

def clean_series_safe(series, batch_size=50000):
    cleaned_list = []
    total_len = len(series)
    for start_idx in range(0, total_len, batch_size):
        batch = series.iloc[start_idx:start_idx + batch_size]
        cleaned_batch = [clean_text_pure_python(x) for x in batch]
        cleaned_list.extend(cleaned_batch)
    return cleaned_list

for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    print(f"Cleaning dataframe securely with PyArrow...")
    name_list = clean_series_safe(df['business_name'])
    addr_list = clean_series_safe(df['business_address'])
    df['clean_name'] = pd.Series(name_list, dtype="string[pyarrow]")
    df['clean_address'] = pd.Series(addr_list, dtype="string[pyarrow]")

print("Text preprocessing completed successfully!")


Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Text preprocessing completed successfully!


In [5]:
import os

# Create an output folder for cleaned data if it doesn't exist
os.makedirs("output/cleaned_data", exist_ok=True)

# Dictionary of dataframes to export
cleaned_dfs = {
    "train_source1": train_s1,
    "train_source2": train_s2,
    "train_source3": train_s3,
    "test_source1": test_s1,
    "test_source2": test_s2,
    "test_source3": test_s3
}

# Export each dataframe to a tab-separated (.tsv) file
for name, df in cleaned_dfs.items():
    output_path = f"output/cleaned_data/{name}_cleaned.tsv"
    df.to_csv(output_path, sep="\t", index=False)
    print(f"Successfully exported: {output_path}")

print("All cleaned databases exported successfully!")

Successfully exported: output/cleaned_data/train_source1_cleaned.tsv
Successfully exported: output/cleaned_data/train_source2_cleaned.tsv
Successfully exported: output/cleaned_data/train_source3_cleaned.tsv
Successfully exported: output/cleaned_data/test_source1_cleaned.tsv
Successfully exported: output/cleaned_data/test_source2_cleaned.tsv
Successfully exported: output/cleaned_data/test_source3_cleaned.tsv
All cleaned databases exported successfully!


In [9]:
dfs_to_fix = [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]

for df in dfs_to_fix:
    for col in df.select_dtypes(include=['string', 'object']).columns:
        df[col] = df[col].astype(str)

print("Dataframes successfully converted to standard string type. Ready to run NearestNeighbors!")

Dataframes successfully converted to standard string type. Ready to run NearestNeighbors!


In [10]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

def generate_candidates_nn(s1_df, s2_df, s3_df, threshold=0.20, chunk_size=2000):
    candidate_pairs = []
    countries = s1_df['country'].dropna().unique()
    
    # Cosine similarity threshold converted to cosine distance threshold:
    # similarity >= 0.20  ==>  distance <= (1.0 - 0.20) = 0.80
    max_distance = 1.0 - threshold
    
    for country in countries:
        s1_sub = s1_df[s1_df['country'] == country]
        s2_sub = s2_df[s2_df['country'] == country]
        s3_sub = s3_df[s3_df['country'] == country]
        
        if s1_sub.empty:
            continue
        pool_df = pd.concat([s2_sub, s3_sub], ignore_index=True)
        if pool_df.empty:
            continue
            
        print(f"Processing country '{country}': {len(s1_sub)} records vs {len(pool_df)} pool records...")
        
        # Fit TF-IDF vectorizer on combined text space
        vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4))
        vectorizer.fit(pd.concat([s1_sub['clean_name'], pool_df['clean_name']]))
        
        tfidf_pool = vectorizer.transform(pool_df['clean_name'])
        
        # Initialize NearestNeighbors using brute-force for sparse matrices
        nn_model = NearestNeighbors(n_neighbors=min(50, len(pool_df)), metric='cosine', algorithm='brute')
        nn_model.fit(tfidf_pool)
        
        # Query in small chunks to control RAM usage completely
        for start_idx in range(0, len(s1_sub), chunk_size):
            s1_chunk = s1_sub.iloc[start_idx:start_idx + chunk_size]
            tfidf_s1_chunk = vectorizer.transform(s1_chunk['clean_name'])
            
            # Find neighbors and distances
            distances, indices = nn_model.kneighbors(tfidf_s1_chunk)
            
            # Filter by distance threshold (cosine distance <= 0.80 corresponds to similarity >= 0.20)
            for r_idx in range(len(s1_chunk)):
                s1_id = s1_chunk.iloc[r_idx]['entity_id']
                for n_idx, dist in zip(indices[r_idx], distances[r_idx]):
                    similarity = 1.0 - dist
                    if similarity >= threshold:
                        candidate_pairs.append({
                            'source1_entity_id': s1_id,
                            'candidate_entity_id': pool_df.iloc[n_idx]['entity_id']
                        })
                        
    return pd.DataFrame(candidate_pairs)

# Re-run candidate generation safely with NearestNeighbors
train_candidates = generate_candidates_nn(train_s1, train_s2, train_s3, threshold=0.20)
print(f"Generated {len(train_candidates)} candidate pairs successfully!")

ArrowMemoryError: malloc of size 144154624 failed